# Gold Layer - Retail Sales Lakehouse

**Purpose:** Create analytics-ready business KPI tables from curated Silver data

---

### Input Tables:
* `silver_sales`: Validated and deduplicated sales transactions

### Output Tables:
* `gold_store_sales`: Store-level sales metrics and performance indicators
* `gold_customer_sales`: Customer-level purchase behavior and segmentation
* `gold_category_sales`: Product category sales analytics

### Job Parameters:
* `minimum_sales`: Threshold for high-value customer classification

---

In [0]:
# Import required libraries
from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("Gold Pipeline Started")

In [0]:
# Read minimum_sales threshold from Lakeflow Job parameter
dbutils.widgets.text("minimum_sales", "")
minimum_sales_param = dbutils.widgets.get("minimum_sales").strip()

if minimum_sales_param == "":
    raise Exception("minimum_sales job parameter is required.")

minimum_sales_threshold = int(minimum_sales_param)

print(f"Minimum Sales Threshold: {minimum_sales_threshold}")

In [0]:
# Read validated silver layer data
silver_sales_df = spark.table("silver_sales")
silver_count = silver_sales_df.count()
print(f"Rows in Silver: {silver_count}")

In [0]:
# Aggregate sales metrics by store
gold_store_sales_df = silver_sales_df.groupBy("store_id").agg(
    F.sum("sales_amount").alias("total_sales"),
    F.count("transaction_id").alias("total_transactions"),
    F.avg("sales_amount").alias("average_order_value"),
    F.countDistinct("customer_id").alias("unique_customers"),
    F.max("sales_amount").alias("highest_sale"),
    F.min("sales_amount").alias("lowest_sale")
)

# Write to Delta
gold_store_sales_df.write.format("delta").mode("overwrite").saveAsTable("gold_store_sales")
store_count = gold_store_sales_df.count()
print(f"Rows in Store Gold: {store_count}")

In [0]:
# Calculate favorite category per customer (category with highest purchase frequency)
customer_category_freq = silver_sales_df.groupBy("customer_id", "product_category").agg(
    F.count("*").alias("purchase_count")
)

# Window to rank categories by purchase count per customer
window_spec = Window.partitionBy("customer_id").orderBy(F.desc("purchase_count"))
customer_favorite_category = customer_category_freq.withColumn(
    "rank", F.row_number().over(window_spec)
).filter(F.col("rank") == 1).select(
    "customer_id",
    F.col("product_category").alias("favorite_category")
)

# Aggregate customer metrics
gold_customer_sales_df = silver_sales_df.groupBy("customer_id").agg(
    F.count("transaction_id").alias("total_orders"),
    F.sum("sales_amount").alias("total_spend"),
    F.avg("sales_amount").alias("average_order_value"),
    F.max("transaction_time").alias("last_purchase_date")
)

# Join with favorite category
gold_customer_sales_df = gold_customer_sales_df.join(
    customer_favorite_category,
    "customer_id",
    "left"
)

# Add high value customer flag based on job parameter
gold_customer_sales_df = gold_customer_sales_df.withColumn(
    "high_value_customer",
    F.when(F.col("total_spend") >= minimum_sales_threshold, True).otherwise(False)
)

# Write to Delta
gold_customer_sales_df.write.format("delta").mode("overwrite").saveAsTable("gold_customer_sales")
customer_count = gold_customer_sales_df.count()
print(f"Rows in Customer Gold: {customer_count}")

In [0]:
# Aggregate sales metrics by product category
gold_category_sales_df = silver_sales_df.groupBy("product_category").agg(
    F.sum("sales_amount").alias("total_sales"),
    F.count("transaction_id").alias("total_transactions"),
    F.avg("sales_amount").alias("average_sale"),
    F.countDistinct("customer_id").alias("unique_customers")
)

# Write to Delta
gold_category_sales_df.write.format("delta").mode("overwrite").saveAsTable("gold_category_sales")
category_count = gold_category_sales_df.count()
print(f"Rows in Category Gold: {category_count}")

print("Gold Pipeline Completed")

In [0]:
# Display all three Gold tables
print("\n=== Gold Store Sales ===")
display(spark.table("gold_store_sales"))

print("\n=== Gold Customer Sales ===")
display(spark.table("gold_customer_sales"))

print("\n=== Gold Category Sales ===")
display(spark.table("gold_category_sales"))

## Enterprise Production Deployment

In a production environment, these Gold layer tables would typically:

### 1. Feed Business Intelligence Tools:
* **Power BI:** Direct query or import mode for executive dashboards
* **Tableau:** Live connection for interactive analytics
* **Databricks SQL Dashboards:** Real-time visualization and reporting

### 2. Serve as the consumption layer for:
* Sales performance monitoring
* Customer segmentation and targeting
* Store performance analysis
* Product category insights

### 3. Be refreshed automatically through Lakeflow Jobs:
* Scheduled daily/hourly refreshes based on business needs
* Triggered by upstream Silver layer completion
* Monitored with job alerts and SLA tracking
* Version controlled with Delta Lake time travel

### 4. Support downstream use cases:
* ML feature engineering for customer lifetime value prediction
* Ad-hoc analysis by business analysts
* API integrations for operational applications
* Regulatory reporting and auditing